# Capture delay calibration

This notebook calibrates QuEL-1 capture timing using `capture_delay` (128 ns steps) and `capture_delay_word` (8 ns per word, in multiples of four words). It centers the complete pulse in the capture window while retaining at least 32 ns of margin on each side. This workflow is for QuEL-1.

Delay settings are restored after each acquisition. Frequencies are restored after each mux sweep. Configuration files are written only when you call `save_params()`. Select `.venv` as the notebook environment and use the repository root as the working directory; see [README](README.md) for installation and [the API guide](capture_delay_tools/README.md) for details.

Captured waveforms are stored in `calibrator.waveforms`. Each `Waveform.iq` is an `xarray.DataArray` with a `time_ns` coordinate. For example, `calibrator.waveforms[0].iq.sel(time_ns=slice(100, 400))` selects a time interval. Analysis and candidate reports are pandas DataFrames.


## 1. Prepare the hardware

Check the configuration path, system ID, and selected muxes before connecting. The values below retain the configuration of the Japanese notebook.


In [ ]:
import os
from pathlib import Path

import qubex as qb

from capture_delay_tools import CaptureDelayCalibrator

# Set QUBEX_CONFIG_ROOT to the configuration repository for this environment.
config_root = (
    Path(os.environ.get("QUBEX_CONFIG_ROOT", "./tmp/qubex-config"))
    .expanduser()
    .resolve()
)
system_id = "64Qv3"
exp = qb.Experiment(
    system_id=system_id,
    config_dir=config_root / system_id / "config",
    params_dir=config_root / system_id / "params",
    # muxes=[0,1,2,3,4,5,6,7,8,9,12,13,14,15],
    muxes=[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15],
)

In [ ]:
exp.connect()

## 2. Sweep capture delays

For each selected mux, the initial sweep measures 19 coarse delays (0 through 18) with word offset zero. Mux indices are resolved from the experiment targets. Edit the sweep ranges below as needed.

Every coarse/word combination is acquired once per mux, with all selected targets captured simultaneously. Three coarse delays and four word offsets therefore require 12 acquisitions per mux. To sweep only words, use one coarse delay and `range(0, 16, 4)` for words. Omitting a mux from `capture_delay_word` retains its configured word offset.

The heatmap shows delay/word pairs on the horizontal axis, time within the capture in ns on the vertical axis, and IQ magnitude as color. The horizontal axis is categorical: adjacent points need not be equally spaced in time. The total delay is `128 * delay + 8 * word` ns.

Each acquisition uses 1024 shots, including verification and the final check. Figures are generated separately for each qubit. Plot methods called with `show=False` return a dictionary of Figures keyed by qubit name.


### Fixed test frequencies

Set `readout_frequencies_ghz` to the absolute test frequency in GHz for every selected mux. Choose values supported by the controller, wiring, and filters; resonator calibration is not required. Muxes with the same signal path characteristics can share a frequency. Missing entries raise an error before acquisition.

Only the first selected target in each mux transmits, using its configured readout amplitude. Other transmit amplitudes are zero, while all selected targets are captured simultaneously at the same frequency. The frequency override spans each mux sweep and is restored afterwards. Sweeps, verification, and final checks use the same fixed frequencies. Frequency YAML files are not modified.


In [ ]:
# Choose frequencies supported by the connected signal path (values in GHz).
# The 11 GHz values below are site settings, not a universal recommendation.
readout_frequencies_ghz = {index: 11.0 for index in range(16)}
readout_frequencies_ghz

In [ ]:
system = exp.system_manager.experiment_system
mux_indices = sorted({system.get_mux_by_qubit(q).index for q in exp.qubit_labels})
if not mux_indices:
    raise ValueError(
        "No targets selected. Check the experiment muxes/qubits and configuration."
    )

delay_sweep = {mux: range(19) for mux in mux_indices}
word_sweep = {mux: [0] for mux in mux_indices}

# Word-only sweep example: replace 7 with a coarse delay selected for your setup.
# delay_sweep = {mux: [7] for mux in mux_indices}
# word_sweep = {mux: range(0, 16, 4) for mux in mux_indices}

calibrator = CaptureDelayCalibrator(
    exp,
    readout_frequencies_ghz=readout_frequencies_ghz,
    n_shots=1024,
    snr_threshold=5.0,
    smoothing=5,
)
display(calibrator.sweep(delay_sweep, capture_delay_word=word_sweep))
calibrator.plot_sweep()

## 3. Analyze edges and select a centered candidate

Red markers indicate rising edges, orange markers indicate falling edges, and the dashed line marks the capture-window center. Pulse midpoints are calculated as follows:

- Both edges detected: `(rise + fall) / 2`.
- Rising edge only: `rise + duration / 2`.
- Falling edge only: `fall - duration / 2`.

The duration is the readout pulse duration in ns recorded during acquisition. Each target needs at least two distinct total-delay settings whose absolute midpoint estimates agree within 64 ns. The complete envelope, from the earliest rise to the latest fall across all targets, is centered using a nonnegative 32 ns aligned coarse/word pair that leaves at least 32 ns of margin on both sides.


In [ ]:
# Optional analysis window in ns: windows={"Q00": (0, 500)}.
# Crop a specific delay/word pair: windows={("Q00", 7, 4): (100, 400)}.
_ = calibrator.analyze()
display(calibrator.select())
calibrator.plot_analysis()

## 4. Sweep word offsets 0, 4, 8, and 12

Keep each selected coarse delay fixed and acquire four word offsets: **0, 4, 8, and 12**. Together with the initial 19-point coarse sweep, this normally takes 23 acquisitions per mux. Coarse results remain in `coarse_calibrator`; a new `calibrator` holds the word sweep.

Use the per-qubit heatmaps to check that signal positions shift in 32 ns steps as the word offset increases. The fine sweep must also provide consistent midpoint estimates before verification.


In [ ]:
coarse_calibrator = calibrator
if not coarse_calibrator.selected_delays:
    display(coarse_calibrator.selection)
    raise ValueError(
        "No coarse-delay candidates. Check the sweep range and selection reasons."
    )

calibrator = CaptureDelayCalibrator(
    exp,
    readout_frequencies_ghz=readout_frequencies_ghz,
    n_shots=1024,
    snr_threshold=5.0,
    smoothing=5,
)
display(
    calibrator.sweep(
        {mux: [delay] for mux, delay in coarse_calibrator.selected_delays.items()},
        capture_delay_word={
            mux: range(0, 16, 4) for mux in coarse_calibrator.selected_delays
        },
    )
)
_ = calibrator.analyze()
display(calibrator.select())
calibrator.plot_analysis()

## 5. Verify the centered candidate

Temporarily apply both `selected_delay` and `selected_capture_delay_word` from `selection`. `midpoint_ns` is the midpoint of the two measured edges; `error_ns` is its displacement from the window center. Even if a candidate was inferred from one edge, verification must detect both edges before finalization can succeed.


In [ ]:
display(calibrator.selected_delays)
display(calibrator.verify())
calibrator.plot_verification()

## 6. Calculate the final coarse and word values

Only targets with both edges detected during verification can support a final result. Center the complete envelope from the earliest rise to the latest fall across all targets. Among nonnegative 32 ns aligned delays that leave at least 32 ns on both sides, select the one closest to the window center. Split the total delay into `capture_delay` (128 ns units) and `capture_delay_word` (remaining 8 ns words).

`predicted_midpoint_ns` reports the predicted envelope midpoint. `predicted_pre_margin_ns` and `predicted_post_margin_ns` report the smallest margins across all targets. One-sided verification does not produce final values. If `pulse_does_not_fit` is reported, review the capture length; also check the sweep ranges and analysis results.


In [ ]:
display(calibrator.finalize())
display(
    {
        "capture_delay": calibrator.final_delays,
        "capture_delay_word": calibrator.final_delay_words,
    }
)

## 7. Inspect the final waveforms

Remeasure using the calculated coarse/word pair. The predicted margins are at least 32 ns on both sides; inspect the I/Q traces to confirm that the complete signal is captured.

If `final_delays` is empty, inspect `final_report.reason`. `verification_failed` means a required target was not verified, `missing_falling_edge` means its falling edge was not detected, and `pulse_does_not_fit` means the pulse cannot fit with the required margins. Repeating a sweep, analysis, selection, or verification clears downstream final results; run `finalize()` again after verification.

Detailed results are in `analysis` and `verification`, with raw data in `waveforms`. Each waveform records `delay`, `capture_delay_word`, and `total_delay_ns`. Final figure captions show the applied pair. Use `calibrator.plot_final()` to redraw stored results without another acquisition.


In [ ]:
# Finalize verified data if the previous finalization cell was skipped.
if not calibrator.final_delays and not calibrator.verification.empty:
    display(calibrator.finalize())

if calibrator.final_delays:
    display(
        {
            "capture_delay": calibrator.final_delays,
            "capture_delay_word": calibrator.final_delay_words,
        }
    )
    calibrator.check_final()
else:
    display(calibrator.selection)
    display(calibrator.verification)
    display(calibrator.final_report)
    if calibrator.final_report.empty:
        print("No verified data. Check the results of select() and verify().")
    else:
        reasons = {
            "verification_failed": "A required rising edge was not detected during verification.",
            "missing_falling_edge": "A falling edge was not detected.",
            "pulse_does_not_fit": "The pulse cannot fit with 32 ns margins.",
        }
        for row in calibrator.final_report.itertuples():
            print(f"MUX{row.mux}: {reasons.get(row.reason, row.reason)}")
    print("No final values were accepted; the final acquisition was skipped.")

## Save final candidates (optional)

Inspect the final waveforms before running the save cell below. Change its destination to the intended parameter directory.

```python
calibrator.save_params("/path/to/params")
# Update existing destination files explicitly:
calibrator.save_params("/path/to/params", overwrite=True)
```

For a new destination, existing files in the experiment's `config_loader.params_path` provide the base. Use `source_dir="/path/to/source/params"` to choose a different base. When overwriting, existing destination files take precedence, individually. A missing base file is created with the native unit and `default: 0`.

The function updates only finalized muxes in `capture_delay.yaml` and `capture_delay_word.yaml`, preserving other muxes, metadata, comments, and quotes. Whitespace and indentation may change. Saving does not reload or modify the active experiment.


In [ ]:
calibrator.save_params("./tmp")

### Analysis environment

`TimeOfFlightAnalysis` calls `sqe-analysis` directly in this repository's Python 3.13 environment. Analysis does not launch a separate process or install dependencies at runtime. Pinned revisions are recorded in `pyproject.toml` and `uv.lock`.


### Rerun after updating

Restart the kernel and create a new `calibrator` from the preparation cells.
